# Field Assignment 01: Online Grocery Fulfillment Performance Review  
**Glorystone Data Analytics**  
**Business Question:** Which stores show the clearest operational friction in online order fulfillment, and what specific actions should Glorystone take in the next 60–90 days?

Setup


In [ ]:
import pandas as pd

Load Data

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/CSRodgers184/Glorystone-DataAnalytics/main/01-online-fulfillment-performance/data/glorystone_online_pickup_orders.csv")

## 3. Initial Inspection

Dataset contains 8,500 orders across 12 stores.  
Key observations from initial load:
- 44 missing `store_id` values
- 590 missing `pickup_datetime` values (treated as no-shows)
- Date columns loaded as object and require conversion to datetime

In [ ]:
print(df.shape)
print(df.dtypes)
df.head()
df.isnull().sum()

(8500, 12)
order_id                    object
store_id                    object
region                      object
order_datetime              object
promised_ready_datetime     object
actual_ready_datetime       object
pickup_datetime             object
items_ordered                int64
items_fulfilled              int64
substitution_count           int64
is_complete                   bool
order_value                float64
dtype: object


,0
order_id,0
store_id,44
region,0
order_datetime,0
promised_ready_datetime,0
actual_ready_datetime,0
pickup_datetime,590
items_ordered,0
items_fulfilled,0
substitution_count,0


This is the result I orginalay get and this is what it means. This means that there is 44 missing store Id's and 590 date Pickup times are missing

In [ ]:
print(df.shape)
print(df.dtypes)
df.head()
df.isnull().sum()

(8500, 12)
order_id                    object
store_id                    object
region                      object
order_datetime              object
promised_ready_datetime     object
actual_ready_datetime       object
pickup_datetime             object
items_ordered                int64
items_fulfilled              int64
substitution_count           int64
is_complete                   bool
order_value                float64
dtype: object


,0
order_id,0
store_id,44
region,0
order_datetime,0
promised_ready_datetime,0
actual_ready_datetime,0
pickup_datetime,590
items_ordered,0
items_fulfilled,0
substitution_count,0


## 4. Convert Date Columns

Date columns were converted from object to datetime so that delays and time-based analysis can be calculated correctly.


In [ ]:
date_cols = ["order_datetime", "promised_ready_datetime", "actual_ready_datetime", "pickup_datetime"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df.dtypes)
df.head()
df.isnull().sum()

order_id                           object
store_id                           object
region                             object
order_datetime             datetime64[ns]
promised_ready_datetime    datetime64[ns]
actual_ready_datetime      datetime64[ns]
pickup_datetime            datetime64[ns]
items_ordered                       int64
items_fulfilled                     int64
substitution_count                  int64
is_complete                          bool
order_value                       float64
dtype: object


,0
order_id,0
store_id,44
region,0
order_datetime,0
promised_ready_datetime,0
actual_ready_datetime,0
pickup_datetime,590
items_ordered,0
items_fulfilled,0
substitution_count,0


## 5. Data Cleaning & Feature Engineering

- Dropped 44 rows with missing `store_id`
- Removed rows where `items_ordered` = 0
- Created key metrics:
  - `ready_delay_min` (actual ready time – promised ready time)
  - `fill_rate` (items fulfilled / items ordered)
  - `is_no_show` (missing pickup datetime)

In [ ]:
# 1. Handle missing store_id
# Decision: drop the 44 rows with missing store_id (small % and we cannot assign a store)
print("Rows before dropping missing store_id:", len(df))
df = df.dropna(subset=["store_id"]).copy()
print("Rows after dropping:", len(df))

# 2. Create core derived metrics
df["ready_delay_min"] = (df["actual_ready_datetime"] - df["promised_ready_datetime"]).dt.total_seconds() / 60
df["fill_rate"] = df["items_fulfilled"] / df["items_ordered"]
df["is_no_show"] = df["pickup_datetime"].isna()

# 3. Quick check
print(df[["ready_delay_min", "fill_rate", "is_no_show"]].describe())
print("\nNo-show rate:", df["is_no_show"].mean().round(3))

Rows before dropping missing store_id: 8500
Rows after dropping: 8456
       ready_delay_min    fill_rate
count      8456.000000  8456.000000
mean         10.252247          inf
std          10.366786          NaN
min           0.000000     0.600000
25%           2.000000     0.875000
50%           8.000000     0.916667
75%          15.000000     1.000000
max          76.000000          inf

No-show rate: 0.069


Fix the zero-item rows

In [ ]:
# Check how many rows have items_ordered == 0
print("Rows with items_ordered == 0:", (df["items_ordered"] == 0).sum())

# Drop them (data entry errors)
df = df[df["items_ordered"] > 0].copy()

# Recalculate fill_rate cleanly
df["fill_rate"] = df["items_fulfilled"] / df["items_ordered"]

print("Rows remaining:", len(df))
print(df[["ready_delay_min", "fill_rate"]].describe())

Rows with items_ordered == 0: 22
Rows remaining: 8434
       ready_delay_min    fill_rate
count      8434.000000  8434.000000
mean         10.253735     0.920394
std          10.360117     0.076406
min           0.000000     0.600000
25%           2.000000     0.875000
50%           8.000000     0.916667
75%          15.000000     1.000000
max          76.000000     1.000000


## 6. Store-Level Performance Summary

In [ ]:
store_summary = (
    df.groupby("store_id")
    .agg(
        orders=("order_id", "count"),
        avg_delay=("ready_delay_min", "mean"),
        median_delay=("ready_delay_min", "median"),
        avg_fill_rate=("fill_rate", "mean"),
        substitution_rate=("substitution_count", lambda x: (x > 0).mean()),
        no_show_rate=("is_no_show", "mean"),
        complete_rate=("is_complete", "mean")
    )
    .round(3)
    .sort_values("avg_delay", ascending=False)
)

store_summary

,orders,avg_delay,median_delay,avg_fill_rate,substitution_rate,no_show_rate,complete_rate
store_id,,,,,,,
GS-09,660,25.202,24.0,0.846,0.200,0.077,0.076
GS-04,653,19.421,19.0,0.863,0.170,0.095,0.107
GS-11,729,12.313,12.0,0.901,0.143,0.077,0.228
GS-06,715,10.587,10.0,0.912,0.113,0.059,0.276
GS-08,706,9.272,8.5,0.921,0.140,0.068,0.292
GS-05,724,8.599,8.0,0.934,0.106,0.073,0.362
GS-12,680,8.150,7.5,0.929,0.110,0.054,0.360
GS-02,711,7.391,6.0,0.938,0.105,0.075,0.395
GS-10,745,6.805,6.0,0.939,0.095,0.055,0.435


## 7. Problem Store Deep Dive (GS-09 & GS-04)

In [ ]:
problem_stores = ["GS-09", "GS-04"]
other_stores = df[~df["store_id"].isin(problem_stores)]

print("=== PROBLEM STORES (GS-09 + GS-04) ===")
print(df[df["store_id"].isin(problem_stores)][["ready_delay_min", "fill_rate", "substitution_count"]].describe().round(2))

print("\n=== ALL OTHER STORES ===")
print(other_stores[["ready_delay_min", "fill_rate", "substitution_count"]].describe().round(2))

=== PROBLEM STORES (GS-09 + GS-04) ===
       ready_delay_min  fill_rate  substitution_count
count          1313.00    1313.00             1313.00
mean             22.33       0.85                0.40
std              14.98       0.08                0.93
min               0.00       0.60                0.00
25%              11.00       0.80                0.00
50%              22.00       0.87                0.00
75%              32.00       0.90                0.00
max              76.00       1.00                4.00

=== ALL OTHER STORES ===
       ready_delay_min  fill_rate  substitution_count
count          7121.00    7121.00             7121.00
mean              8.03       0.93                0.20
std               7.34       0.07                0.65
min               0.00       0.67                0.00
25%               1.00       0.88                0.00
50%               7.00       0.93                0.00
75%              12.00       1.00                0.00
max              

check one more cut — performance by hour of day for the problem stores:

In [ ]:
df["order_hour"] = df["order_datetime"].dt.hour

hourly = (
    df[df["store_id"].isin(problem_stores)]
    .groupby("order_hour")["ready_delay_min"]
    .agg(["count", "mean", "median"])
    .round(1)
)

hourly

,count,mean,median
order_hour,,,
0,11,20.7,20.0
1,8,26.0,24.0
2,14,19.3,13.0
3,14,22.0,14.5
4,9,21.1,26.0
5,19,22.7,21.0
6,7,22.7,26.0
7,39,21.5,20.0
8,63,17.6,16.0


## 8. Recommendations

**Primary Finding**  
GS-09 and GS-04 are the clear outliers. They run 14–17 minutes slower on average readiness delay and show meaningfully lower fill rates and higher substitution rates than the rest of the network.

**Priority Actions (next 60–90 days)**  
1. Conduct a focused on-hand accuracy audit at GS-09 and GS-04 for the highest-volume and highest-substitution items. Correct inventory records and adjust replenishment settings where systemic discrepancies are found.  
2. Pull a substitution detail report for both stores to identify the specific items most frequently substituted. Prioritize those items for inventory correction and supplier reliability checks.  
3. Hold a structured working session with the Team Leads at GS-09 and GS-04. Share the delay and fill-rate data, set clear readiness targets, and surface the operational obstacles they are seeing on the floor.

**Expected Impact**  
Improving on-hand accuracy should raise fill rates and reduce substitution volume. Combined with clearer targets and input from the store teams, this is expected to bring average readiness delay at these two stores closer to the network average within 60–90 days.

Notes for Myself

When You look at this analysis you should see: What the data is telling us (clear signal)

Operational friction is highly concentrated in GS-09 and GS-04.

*   These two stores run ~14–17 minutes slower on average than the rest of the network.

*    GS-09- MEdian delay : 24
*    GS-04- Median delay : 19 minutes


*   They also have meaningfully lower fill rates and higher substitution rates.
*   The delay problem is not limited to a single peak hour — it is elevated across most of the operating day, with noticeable spikes in late morning and late afternoon.

This is a clean, decision-relevant finding.